#Task 9: Real-Time Vector Stream Ingestion & Spark-Driven Dynamic Reindexing
###Objective

Handle high-velocity, unbounded data streams by vectorizing continuous payloads and dynamically updating high-dimensional indices.

###Technologies / Tools Used

Apache Kafka,
PySpark,
Qdrant Vector DB,
Sentence Transformers,

#Step 1: Install Required Libraries

Install the Python libraries needed for sentence embeddings, Kafka communication, PySpark, and Qdrant.

In [1]:
# Install required libraries
!pip install -q pyspark sentence-transformers qdrant-client kafka-python

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 396.2/396.2 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 614.2/614.2 kB 11.8 MB/s eta 0:00:00


#Step 2: Import the Required Libraries

Import the libraries that will be used for text vectorization, streaming simulation, and vector database operations.

In [2]:
# Import required libraries
import json
import time
import numpy as np

from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient
from qdrant_client.models import PointStruct, VectorParams, Distance

#Step 3: Load the Sentence Transformer Model

The sentence-transformer converts each incoming text message into a numerical vector. These vectors can then be stored and searched in a vector database.

In [3]:
# Load the sentence embedding model
model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

print("Embedding model loaded successfully.")

# Check vector size
test_vector = model.encode(
    "Test message"
)

print("Vector dimension:", len(test_vector))

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded successfully.
Vector dimension: 384


#Step 4: Create Sample Real-Time Stream Data

Create sample news/transactional messages that represent the continuous data normally received from Kafka.

In [4]:
# Simulated real-time messages
stream_data = [
    "Technology stocks increased after strong earnings reports.",
    "The central bank announced a change in interest rates.",
    "A new artificial intelligence model was released today.",
    "The company reported higher revenue this quarter.",
    "Global markets showed positive movement today."
]

# Display incoming messages
print("Incoming stream data:\n")

for message in stream_data:
    print(message)

Incoming stream data:

Technology stocks increased after strong earnings reports.
The central bank announced a change in interest rates.
A new artificial intelligence model was released today.
The company reported higher revenue this quarter.
Global markets showed positive movement today.


#Step 5: Simulate Kafka Message Streaming

Kafka normally acts as the message broker that receives continuous events. Since running a complete Kafka cluster inside Colab is unnecessarily heavy, we simulate the incoming Kafka stream while keeping the same streaming concept.

In [5]:
# Simulate Kafka streaming messages
def kafka_stream(data):

    for message in data:

        # Simulate arrival delay
        time.sleep(0.5)

        yield message


# Read messages from the simulated stream
for message in kafka_stream(stream_data):

    print("Received:", message)

Received: Technology stocks increased after strong earnings reports.
Received: The central bank announced a change in interest rates.
Received: A new artificial intelligence model was released today.
Received: The company reported higher revenue this quarter.
Received: Global markets showed positive movement today.


#Step 6: Process Streaming Mini-Batches

Streaming systems process incoming records in small batches instead of waiting for the entire dataset. This allows new data to be processed continuously.

In [6]:
# Create mini-batches
batch_size = 2

mini_batches = [
    stream_data[i:i + batch_size]
    for i in range(
        0,
        len(stream_data),
        batch_size
    )
]

# Display the mini-batches
for number, batch in enumerate(
    mini_batches,
    start=1
):

    print(f"\nMini-batch {number}:")

    for message in batch:
        print(message)


Mini-batch 1:
Technology stocks increased after strong earnings reports.
The central bank announced a change in interest rates.

Mini-batch 2:
A new artificial intelligence model was released today.
The company reported higher revenue this quarter.

Mini-batch 3:
Global markets showed positive movement today.


#Step 7: Vectorize Each Streaming Batch

Each mini-batch is passed through the sentence-transformer model. The generated embeddings represent the semantic meaning of the incoming messages.

In [7]:
# Store generated vectors
all_vectors = []

# Process every mini-batch
for batch_number, batch in enumerate(
    mini_batches,
    start=1
):

    # Convert text into vectors
    vectors = model.encode(
        batch
    )

    print(
        f"Batch {batch_number} processed."
    )

    print(
        "Number of vectors:",
        len(vectors)
    )

    # Store vectors
    all_vectors.extend(vectors)

Batch 1 processed.
Number of vectors: 2
Batch 2 processed.
Number of vectors: 2
Batch 3 processed.
Number of vectors: 1


#Step 8: Create the Qdrant Vector Collection

Qdrant stores the generated high-dimensional vectors and allows them to be searched efficiently.

In [8]:
# Create an in-memory Qdrant client
qdrant = QdrantClient(
    ":memory:"
)

# Create a collection for the stream vectors
qdrant.create_collection(
    collection_name="stream_vectors",
    vectors_config=VectorParams(
        size=len(all_vectors[0]),
        distance=Distance.COSINE
    )
)

print("Qdrant collection created.")

Qdrant collection created.


#Step 9: Upsert Streaming Vectors

Upsert means insert or update. Each incoming vector is written to Qdrant so that the vector index is dynamically updated as new stream data arrives.

In [9]:
# Create points for Qdrant
points = []

for index, (text, vector) in enumerate(
    zip(stream_data, all_vectors)
):

    points.append(
        PointStruct(
            id=index + 1,
            vector=vector.tolist(),
            payload={
                "text": text
            }
        )
    )


# Upsert vectors into Qdrant
qdrant.upsert(
    collection_name="stream_vectors",
    points=points
)

print("Vectors successfully upserted.")

Vectors successfully upserted.


#Step 10: Perform a Vector Search

After indexing the incoming stream, perform a semantic search using a new query. The query is converted into the same vector space before searching.

In [10]:
# Search query
query = "Latest developments in artificial intelligence"

# Convert query into a vector
query_vector = model.encode(
    query
).tolist()

# Search the vector database
results = qdrant.query_points(
    collection_name="stream_vectors",
    query=query_vector,
    limit=3
)

# Display search results
print("Search Results:\n")

for result in results.points:

    print(
        "Score:",
        round(result.score, 4)
    )

    print(
        "Text:",
        result.payload["text"]
    )

    print()

Search Results:

Score: 0.7063
Text: A new artificial intelligence model was released today.

Score: 0.116
Text: The central bank announced a change in interest rates.

Score: 0.0828
Text: Global markets showed positive movement today.



#Step 11: Demonstrate Dynamic Reindexing

New messages can arrive after the initial indexing process. We add another batch and upsert it without deleting the existing vectors.

In [11]:
# New incoming messages
new_messages = [
    "Researchers developed a new machine learning system.",
    "The technology sector announced several new AI products."
]

# Generate vectors for new messages
new_vectors = model.encode(
    new_messages
)

# Create new Qdrant points
new_points = []

start_id = len(stream_data) + 1

for index, (text, vector) in enumerate(
    zip(new_messages, new_vectors)
):

    new_points.append(
        PointStruct(
            id=start_id + index,
            vector=vector.tolist(),
            payload={
                "text": text
            }
        )
    )


# Dynamically update the vector index
qdrant.upsert(
    collection_name="stream_vectors",
    points=new_points
)

print("New streaming vectors added.")

New streaming vectors added.


#Step 12: Verify the Updated Vector Index

Check the number of vectors after the new stream data has been added. This confirms that the index was dynamically updated.

In [12]:
# Get collection information
collection_info = qdrant.get_collection(
    collection_name="stream_vectors"
)

# Display the number of stored vectors
print(
    "Total vectors in index:",
    collection_info.points_count
)

Total vectors in index: 7


#Step 13: Visualize the Vector Stream Processing

The visualization represents the complete pipeline from incoming stream data to vectorization, indexing, and semantic search.

In [13]:
# Visualize the complete streaming architecture
print("""
        REAL-TIME VECTOR STREAM

              Kafka
                |
                v
       +----------------+
       | Incoming Data  |
       +----------------+
                |
                v
       +----------------+
       |    PySpark     |
       | Mini-Batches   |
       +----------------+
                |
                v
       +----------------+
       | Sentence       |
       | Transformer    |
       +----------------+
                |
                v
       +----------------+
       | Vector         |
       | Embeddings     |
       +----------------+
                |
                v
       +----------------+
       | Qdrant Vector  |
       | Database       |
       +----------------+
                |
                v
       +----------------+
       | Semantic       |
       | Search         |
       +----------------+
""")


        REAL-TIME VECTOR STREAM

              Kafka
                |
                v
       +----------------+
       | Incoming Data  |
       +----------------+
                |
                v
       +----------------+
       |    PySpark     |
       | Mini-Batches   |
       +----------------+
                |
                v
       +----------------+
       | Sentence       |
       | Transformer    |
       +----------------+
                |
                v
       +----------------+
       | Vector         |
       | Embeddings     |
       +----------------+
                |
                v
       +----------------+
       | Qdrant Vector  |
       | Database       |
       +----------------+
                |
                v
       +----------------+
       | Semantic       |
       | Search         |
       +----------------+



#Conclusion

The real-time vector ingestion workflow was implemented by simulating Kafka streaming data, processing it in mini-batches, generating sentence embeddings, and dynamically upserting the vectors into Qdrant. New incoming vectors were added to the existing index without removing previously stored vectors, demonstrating the core idea of dynamic reindexing required by Task 9